# 🛠 1. Подготовка фундамента: Данные и инструменты
Прежде чем строить карту валютного рынка, необходимо подготовить «стерильную» среду и загрузить актуальные котировки. В этом разделе мы подключаем аналитический стек Python и импортируем базу **абсолютных курсов Abscur**. 

**Что именно мы делаем:**
1. **Загружаем библиотеки:** Используем `Pandas` для работы с таблицами и `Seaborn/Matplotlib` для будущей визуализации.
2. **Настраиваем горизонт:** Мы ограничиваем анализ последними **3 годами**. Это позволяет отсечь неактуальные исторические шумы и сфокусироваться на тех рыночных трендах, которые определяют реальность сегодня.
3. **Очищаем данные:** Удаляем валюты-пустышки и заполняем технические пропуски, чтобы волатильность была «чистой», а не вызванной дырами в данных.
4. **Переходим к доходностям:** Превращаем цены в логарифмические доходности — это стандарт финансовой математики, позволяющий корректно сравнивать активы с разной стоимостью.

In [1]:
# =================================================================
# 1. ENVIRONMENT & DATA: ПОДГОТОВКА СРЕДЫ И ЗАГРУЗКА ДАННЫХ
# =================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка визуализации для блога (чистый стиль)
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 12

# --- 1.1. Загрузка данных (Data Ingestion) ---

file_path = '/kaggle/input/notebooks/eavprog/abscur2/abscur.csv'
df_raw = pd.read_csv(file_path, parse_dates=['Date'])
df_raw = df_raw.sort_values('Date').set_index('Date')

# --- 1.2. Выбор горизонта (Настройка актуальности) ---

# Согласно стратегии микро-расчетов, берем последние 3 года 
# для отражения текущей рыночной реальности (SEO-актуальность)
end_date = df_raw.index.max()
start_date = end_date - pd.DateOffset(years=3) 
df_prices = df_raw.loc[start_date:end_date].copy()

# --- 1.3. Очистка и фильтрация (Data Cleaning) ---

# Удаляем активы, у которых слишком мало данных за этот период (новые или удаленные валюты)
# 252 торговых дня в году * 2.5 года (запас на пропуски)
min_required_days = 252 * 2.5 
df_prices = df_prices.dropna(axis=1, thresh=min_required_days)

# Заполнение технических лагов (выходные, праздники)
df_prices = df_prices.ffill().bfill()

# --- 1.4. Расчет логарифмических доходностей (Returns) ---

# Фундамент для расчета CAGR (доходности) и Volatility (риска)
df_returns = np.log(df_prices / df_prices.shift(1)).dropna()

# --- Контроль качества (Quality Check) ---
print(f"✅ Анализ рынка за период: {start_date.date()} — {end_date.date()}")
print(f"✅ Количество анализируемых валют: {df_returns.shape[1]}")
print(f"✅ Размер выборки (дней): {df_returns.shape[0]}")

if df_returns.isnull().values.any():
    print("⚠️ Внимание: обнаружены пропуски в доходностях!")

# Вывод первых строк для визуальной проверки
df_returns.head()

✅ Анализ рынка за период: 2023-03-25 — 2026-03-25
✅ Количество анализируемых валют: 45
✅ Размер выборки (дней): 1011


,AED,ARS,AUD,BRL,CAD,CHF,CLP,CNY,COP,CZK,...,SAR,SEK,SGD,THB,TRY,TWD,UAH,USD,VND,ZAR
Date,,,,,,,,,,,,,,,,,,,,,
2023-03-26,-0.000004,-0.000004,-0.000702,-0.000004,0.000182,0.000405,-0.000004,0.000001,-0.000004,0.000001,...,-0.000004,0.000514,-0.000582,-0.000004,-0.000004,-0.000004,-0.000004,-0.000004,-0.000004,0.000001
2023-03-27,-0.001049,-0.010062,0.000775,0.009462,0.002072,0.002453,0.004850,-0.002965,0.012396,-0.001296,...,-0.001325,0.002975,0.000409,-0.006718,-0.003345,-0.001500,-0.001005,-0.001005,-0.000665,-0.009647
2023-03-28,-0.003573,-0.005494,0.003860,0.002018,0.000128,-0.007271,0.007178,-0.002801,-0.002812,0.007317,...,-0.003439,-0.001834,-0.001074,0.001756,-0.005130,-0.003881,-0.003519,-0.003519,-0.002667,0.005760
2023-03-29,0.000311,-0.002029,-0.002587,0.006926,-0.001963,0.001923,0.005035,-0.001447,0.010938,0.002671,...,0.000869,-0.003210,-0.001070,0.001955,-0.002028,-0.006242,0.000230,0.000230,0.000230,0.002688
2023-03-30,-0.002500,-0.003714,0.001261,0.005009,0.000409,0.002363,0.002321,-0.000198,-0.008250,0.005137,...,-0.002098,0.001204,-0.001586,-0.000631,-0.003352,0.000073,-0.002418,-0.002418,-0.001992,0.013093


На основе полученных промежуточных результатов выполнения первого этапа можно сделать следующие выводы для фиксации в тетрадке:

### 📊 Технический анализ выборки (2023–2026)

1. **Актуальность данных:**
   * Сформирована выборка за последние **3 полных года** (1011 торговых дней). Это оптимальный горизонт для микро-расчета: он достаточно продолжителен для статистической надежности, но при этом отражает современную рыночную конъюнктуру (пост-ковидный период, высокие ставки, текущие геополитические циклы).
   
2. **Полнота покрытия:**
   * В расчете участвуют **45 валют**. Фильтр `min_required_days` подтвердил, что все основные мировые и региональные валюты (от AED до ZAR) имеют достаточную плотность данных в абсолютных курсах для корректного сравнения на этом интервале.

3. **Характер доходностей (Log Returns):**
   * Судя по `head()`, матрица содержит стационарные ряды данных. Значения логарифмических доходностей колеблются в малых диапазонах (в среднем от -0.01 до +0.01), что характерно для валютных рынков.
   * Видны всплески волатильности (например, по **ARS** и **BRL** 27 марта 2023 года), что уже на этапе подготовки данных подтверждает наличие на рынке активов с разным риск-профилем для нашей будущей карты.

4. **Готовность к этапу Metrics Engine:**
   * Данные очищены, пропуски заполнены (`ffill/bfill`), аномалий в виде `NaN` не обнаружено. 
   * Текущая матрица `df_returns` является идеальным фундаментом для расчета CAGR и годовой волатильности.

# 📈 2. Переход к доходностям: Почему логарифмы лучше процентов?
На этом этапе мы превращаем «сырые» котировки в математически корректные данные для анализа. Вместо обычного процентного изменения мы рассчитываем **логарифмические доходности** ($r = \ln(P_t / P_{t-1})$).

**Зачем это нужно:**
1. **Аддитивность:** Логарифмические доходности можно просто складывать. Если валюта упала на 10% и выросла на 10%, в обычных процентах вы останетесь в минусе, а в логарифмах — вернетесь в ноль. Это критически важно для расчета среднегодовой доходности (CAGR).
2. **Симметрия:** Они одинаково обрабатывают рост и падение, что делает расчет волатильности (нашего «Риска») более точным и несмещенным.
3. **Стационарность:** Мы переходим от бесконечно растущих графиков цен к стабильным рядам данных, колеблющимся вокруг нуля. Это позволяет применять к валютам методы классической статистики.

**Результат:** Мы получаем «чистое топливо» для нашего Metrics Engine, очищенное от искажений масштаба цен.

In [2]:
# =================================================================
# 2. RETURNS CALCULATION: ПЕРЕХОД К ДОХОДНОСТЯМ
# =================================================================

# Расчет ежедневных логарифмических доходностей
# Формула: r = ln(P_t / P_{t-1})
df_returns = np.log(df_prices / df_prices.shift(1)).dropna()

# Проверка корректности расчета
print(f"✅ Расчет доходностей завершен.")
print(f"✅ Количество временных интервалов: {len(df_returns)}")
print(f"✅ Базовые статистики (первые 5 активов):")
display(df_returns.iloc[:, :5].describe().loc[['mean', 'std', 'min', 'max']])

# Вывод для визуального контроля
df_returns.iloc[:5, :5]

✅ Расчет доходностей завершен.
✅ Количество временных интервалов: 1011
✅ Базовые статистики (первые 5 активов):


,AED,ARS,AUD,BRL,CAD
mean,0.000060,-0.001835,0.000109,0.000064,0.000046
std,0.002203,0.025963,0.003748,0.005780,0.002281
min,-0.009894,-0.767800,-0.035907,-0.029076,-0.008146
max,0.012805,0.052536,0.028148,0.026763,0.018125


,AED,ARS,AUD,BRL,CAD
Date,,,,,
2023-03-26,-0.000004,-0.000004,-0.000702,-0.000004,0.000182
2023-03-27,-0.001049,-0.010062,0.000775,0.009462,0.002072
2023-03-28,-0.003573,-0.005494,0.003860,0.002018,0.000128
2023-03-29,0.000311,-0.002029,-0.002587,0.006926,-0.001963
2023-03-30,-0.002500,-0.003714,0.001261,0.005009,0.000409


Анализ полученных данных подтверждает готовность выборки к построению карты риск-доходность. Вот основные выводы по результатам этапа **Returns Calculation**:

### 📊 Анализ матрицы логарифмических доходностей

1.  **Масштаб волатильности (std):**
    * Данные наглядно демонстрируют разброс рисков. У стабильных валют (например, **AED**, **CAD**) стандартное отклонение находится в пределах **0.0022**. 
    * В то же время у **ARS** (аргентинское песо) волатильность в 10 раз выше (**0.0259**), что сразу закладывает основу для формирования правого (высокорискового) края нашей будущей карты.

2.  **Экстремальные события (min/max):**
    * Значение `min` для **ARS** (-0.7678) указывает на наличие разовых катастрофических обвалов (девальваций) внутри трехлетнего окна. 
    * Для большинства других валют экстремумы симметричны и умеренны (в пределах 1-3%), что говорит о здоровой рыночной динамике, пригодной для статистического усреднения.

3.  **Средние значения (mean):**
    * Среднедневные доходности ожидаемо близки к нулю (например, **0.000060** у AED), что является нормой для лог-доходностей. 
    * Однако даже такие малые значения при годовом масштабировании (на следующем этапе) превратятся в значимые показатели CAGR, которые позволят нам ранжировать активы по эффективности.

4.  **Статистическая плотность:**
    * Количество интервалов (1011) достаточно для того, чтобы стандартное отклонение (`std`) было устойчивым показателем риска, а не случайным шумом. Это критически важно для корректного разделения рынка на квадранты.

---
**Итог:** База данных `df_returns` полностью верифицирована. Мы имеем как «консервативные якоря», так и «высоковолатильные маркеры», что гарантирует репрезентативность итоговой визуализации.

# ⚙️ 3. Двигатель метрик: Оцифровка скорости и риска
На этом этапе мы превращаем массив данных в конкретные координаты. Каждая валюта получает две ключевые характеристики, которые позволят нам сравнить их между собой на одной плоскости.

**Что мы рассчитываем:**
1.  **CAGR (Среднегодовая доходность):** Это «чистая скорость» валюты. Мы смотрим, на сколько процентов в среднем рос или падал актив каждый год в течение последних 3 лет. Это более честный показатель, чем простая средняя доходность, так как он учитывает эффект накопления.
2.  **Годовая волатильность (Риск):** Это показатель «тряски». Мы берем стандартное отклонение ежедневных изменений и приводим его к годовому масштабу (умножая на $\sqrt{252}$). Чем выше это число, тем непредсказуемее ведет себя валюта.

**Зачем здесь медиана?**
Вместо того чтобы сравнивать валюты с абстрактным нулем, мы находим **медиану** рынка по обоим показателям. Это создает «перекрестие» на нашей будущей карте:
* **Медианная доходность** отсекает те активы, которые растут хуже рынка.
* **Медианный риск** показывает средний уровень тревожности в мировой финансовой системе.

**Результат:** Мы получаем готовую таблицу координат, где уже видны первые лидеры — те, кто умудряется обгонять рынок, не превышая средний уровень риска.

In [3]:
# =================================================================
# 3. METRICS ENGINE: РАСЧЕТ CAGR И ВОЛАТИЛЬНОСТИ
# =================================================================

# 1. Расчет среднегодовой доходности (CAGR)
# Суммируем все лог-доходности и масштабируем их к году (252 торговых дня)
total_days = len(df_returns)
annualization_factor = 252 / total_days
cagr = df_returns.sum() * annualization_factor

# 2. Расчет годовой волатильности (Annualized Volatility)
# Стандартное отклонение ежедневных данных, масштабированное на корень из 252
volatility = df_returns.std() * np.sqrt(252)

# 3. Сборка финальной таблицы метрик
df_metrics = pd.DataFrame({
    'Return': cagr,
    'Risk': volatility
})

# 4. Расчет медианных значений (оси для будущих квадрантов)
median_return = df_metrics['Return'].median()
median_risk = df_metrics['Risk'].median()

# Контрольный вывод
print(f"✅ Метрики рассчитаны для {len(df_metrics)} активов.")
print(f"📍 Медианная доходность рынка (CAGR): {median_return:.2%}")
print(f"📍 Медианный риск рынка (Vol): {median_risk:.2%}")

# Вывод ТОП-10 самых доходных валют текущего периода
display(df_metrics.sort_values('Return', ascending=False).head(10))

✅ Метрики рассчитаны для 45 активов.
📍 Медианная доходность рынка (CAGR): 1.62%
📍 Медианный риск рынка (Vol): 5.14%


,Return,Risk
COP,0.076424,0.101334
PLN,0.057503,0.060637
CHF,0.053102,0.052910
ILS,0.050721,0.043764
ISK,0.044927,0.058996
MYR,0.043413,0.051441
SEK,0.043403,0.064117
GBP,0.037359,0.040635
PEN,0.036409,0.051714
NOK,0.034102,0.072477


Анализ полученных метрик позволяет увидеть реальное распределение сил на валютном рынке за последние 3 года. Эти цифры станут фундаментом для нашей карты и деления на квадранты.

### 📊 Анализ расчетных метрик (Metrics Engine)

1. **«Точка равновесия» рынка (Медианы):**
   * **Медианная доходность (1.62%)**: Это важный психологический и экономический порог. Половина мировых валют в абсолютном выражении растет медленнее этой отметки. Все, что выше — уже претендует на «Альфу».
   * **Медианный риск (5.14%)**: Стандартная годовая «тряска» для мировой корзины. Это наш фильтр: активы с риском ниже 5% мы будем считать консервативными, выше — агрессивными.

2. **Лидеры по доходности (Top Efficiency):**
   * **COP (Колумбийское песо):** Абсолютный лидер по CAGR (**7.6%**), но и самый рискованный актив в топ-10 (**10.1%**). Типичный представитель квадранта «Спекуляций».
   * **PLN, CHF, ILS:** Эти валюты показывают доходность в районе **5%**. Обратите внимание на **ILS (израильский шекель)**: его риск (**4.3%**) ниже медианного, что делает его «идеальным кандидатом» в квадрант **Alpha**.
   * **CHF (швейцарский франк):** Демонстрирует уникальный баланс — доходность (5.3%) практически равна риску (5.2%). Это эталон эффективности.

3. **География «Альфы»:**
   * В топ-10 попали как развитые экономики (**GBP, SEK, NOK**), так и развивающиеся (**MYR, PEN**). Это подтверждает твой тезис: в абсолютных координатах «характер» валюты важнее её географии. 
   * Тот факт, что **MYR (малайзийский ринггит)** находится прямо на границе медианного риска (5.14%), делает его интересным пограничным объектом для анализа.

4. **Аномалии:**
   * **NOK (норвежская крона):** При доходности 3.4% имеет аномально высокий риск (7.2%). Она явно проигрывает по эффективности тому же **GBP** (доходность выше, риск почти в два раза ниже).

---
**Итог:** У нас есть четкие координаты. Медианы (1.62% и 5.14%) разрезают рынок на четыре зоны. Мы видим, что «Альфа» существует — есть активы, которые зарабатывают в 3-4 раза больше медианы, не превышая средний рыночный риск.

# ⚓ 4. Установка маяков: Доллар и Золото как мировые эталоны
Чтобы понять, насколько эффективна та или иная валюта, нам нужны признанные во всем мире «мерила стоимости». В этом разделе мы добавляем на нашу карту двух главных тяжеловесов: **Доллар США (USD)** и **Золото (XAU)**.

**Почему это важно:**
1.  **Доллар (USD):** Это базовый защитный актив. Если валюта на нашей карте находится ниже или правее доллара, значит, хранить в ней деньги было бессмысленно — вы получили либо больше риска, либо меньше доходности, чем в «старом добром» баксе.
2.  **Золото (Gold):** Это исторический стандарт ценности. Мы специально подкачиваем данные с биржи Yahoo Finance и пересчитываем их в **абсолютный курс**. Это позволяет нам увидеть «чистое золото» на той же шкале, что и обычные бумажные деньги.

In [5]:
# =================================================================
# 4. BENCHMARKS PREP: ПОДГОТОВКА ОРИЕНТИРОВ (USD & GOLD)
# =================================================================

import yfinance as yf

# 1. Загрузка рыночных данных Золота (XAU/USD)
# Явно указываем auto_adjust и multi_level для совместимости с новыми версиями yf
gold_data = yf.download("GC=F", start=start_date, end=end_date, auto_adjust=True)

if not gold_data.empty:
    # Выбираем колонку Close (в новых версиях yf это может быть Series или DataFrame)
    gold_close = gold_data['Close']
    if isinstance(gold_close, pd.DataFrame):
        gold_close = gold_close.iloc[:, 0]

    # 2. Преобразование Золота в абсолютный курс
    # Синхронизируем по датам (используем inner join через pd.concat для надежности)
    combined = pd.concat([gold_close, df_prices['USD']], axis=1, join='inner').dropna()
    combined.columns = ['Gold_USD', 'Abs_USD']
    
    abs_gold = combined['Gold_USD'] * combined['Abs_USD']

    # 3. Расчет доходностей для Золота
    gold_returns = np.log(abs_gold / abs_gold.shift(1)).dropna()

    if len(gold_returns) > 0:
        # 4. Вычисление метрик для Бентчмарков
        gold_metrics = pd.Series({
            'Return': gold_returns.sum() * (252 / len(gold_returns)),
            'Risk': gold_returns.std() * np.sqrt(252)
        }, name='GOLD (Abs)')
        
        # Для Доллара (уже есть в df_metrics)
        usd_metrics = df_metrics.loc['USD'].copy()
        usd_metrics.name = 'USD (Abs)'

        # 5. Сборка таблицы ориентиров
        df_benchmarks = pd.DataFrame([usd_metrics, gold_metrics])
        
        print("✅ Метрики ориентиров подготовлены:")
        display(df_benchmarks)
    else:
        print("❌ Ошибка: Недостаточно данных для расчета доходности золота.")
else:
    print("❌ Ошибка: Данные по золоту не загружены. Проверьте соединение или тикер.")

[*********************100%***********************]  1 of 1 completed

✅ Метрики ориентиров подготовлены:


,Return,Risk
USD (Abs),0.015242,0.034920
GOLD (Abs),0.293065,0.184343


Анализ полученных бенчмарков вскрывает крайне интересную рыночную ситуацию за последние 3 года. Эти две точки станут нашими главными навигационными огнями на будущей карте.

### 📊 Анализ ориентиров: USD vs GOLD (Absolute)

1. **Доллар США (USD): Консервативный эталон**
   * **Доходность (1.52%):** Находится практически на уровне медианы рынка (1.62%). Это подтверждает, что в абсолютном выражении доллар за последние 3 года не является «ракетой», а движется в фарватере мировой экономики.
   * **Риск (3.49%):** Значительно ниже медианного риска рынка (5.14%). Это математическое подтверждение статуса доллара как защитного актива: он медленный, но очень стабильный.

2. **Золото (GOLD): Агрессивный лидер**
   * **Доходность (29.31%):** Показывает феноменальный результат в абсолютных координатах. Оно обгоняет среднюю валюту почти в 18 раз. 
   * **Риск (18.43%):** Золото ведет себя не как «тихая гавань», а как высокорисковый актив. Его волатильность в 5.3 раза выше, чем у доллара. На карте оно улетит далеко в правый верхний угол (квадрант Спекуляций/Роста).

### 💡 Ключевые выводы для карты:
* **Граница «Альфы»:** Любая валюта, которая на нашей карте окажется **левее USD** (риск < 3.49%) и **выше USD** (доходность > 1.52%), будет считаться «улучшенным долларом» на текущем отрезке времени.
* **Масштаб:** Золото задает очень высокую планку по оси Y. Это может «прижать» большинство валют к нижней части графика, что наглядно покажет, насколько сильно металлы доминировали над фиатными деньгами в этот период.
* **Абсолютная реальность:** Тот факт, что доходность золота (29%) настолько выше его риска (18%), дает ему выдающийся коэффициент доходность/риск (Sharpe-like ratio), недосягаемый для большинства валют.

# Market Statistics: Определение медианы по доходности и медианы по риску для всей совокупности валют.

# Quadrant Labeling: Присвоение каждой валюте категории (Alpha, Speculative, Conservative, Stagnant) на основе медиан.

# Core Visualization: Построение Scatter-plot (Seaborn), где X — Риск, Y — Доходность. Annotation Layer: Нанесение тикеров валют на точки и отрисовка осевых линий по медианам.

# Insights Table: Вывод ТОП-5 валют из зоны «Alpha» (максимальная доходность при риске ниже среднего).